# Превью датасета U-Net (yaml → irt_data)

Загружает `segmentation/U-Net/dataset_tsr.yaml` (или другой yaml) и показывает много кропов 128×128 с GT-масками.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path("..").resolve()
UNET = ROOT / "segmentation" / "U-Net"
for p in (ROOT, UNET):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from data import make_loader

YAML = UNET / "dataset_tsr.yaml"   # или dataset_fourier.yaml / dataset.yaml
N_SHOW = 24                        # сколько примеров на сетке
SEED = 0

cfg, ds, _ = make_loader(YAML, train=False, size=128)
print(f"yaml={YAML.name} | n={len(ds)} | channels={ds[0][0].shape[0]} | size={tuple(ds[0][0].shape[-2:])}")
print(f"features={cfg.features.extractors} | crop={cfg.crop.strategy} {cfg.crop.size}")


In [ ]:
rng = np.random.default_rng(SEED)
idxs = rng.choice(len(ds), size=min(N_SHOW, len(ds)), replace=False)

ncols = 4
nrows = int(np.ceil(len(idxs) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows))
axes = np.atleast_2d(axes)

for ax, i in zip(axes.ravel(), idxs):
    img, mask = ds[int(i)]
    ch0 = img[0].numpy()
    m = mask[0].numpy()
    ax.imshow(ch0, cmap="inferno")
    ax.contour(m, levels=[0.5], colors="cyan", linewidths=1.2)
    ax.set_title(f"#{int(i)}  mask={m.sum():.0f}px", fontsize=9)
    ax.axis("off")

for ax in axes.ravel()[len(idxs):]:
    ax.axis("off")

fig.suptitle(f"{YAML.name}: channel0 + mask contour (n={len(idxs)})", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# input | mask | overlay — построчно
n = min(8, len(ds))
idxs2 = rng.choice(len(ds), size=n, replace=False)

fig, axes = plt.subplots(n, 3, figsize=(9, 2.4 * n))
if n == 1:
    axes = np.atleast_2d(axes)

for row, i in enumerate(idxs2):
    img, mask = ds[int(i)]
    ch0 = img[0].numpy()
    m = mask[0].numpy()
    overlay = np.stack([ch0, ch0, ch0], axis=-1)
    overlay = (overlay - overlay.min()) / (np.ptp(overlay) + 1e-8)
    overlay[m > 0.5, 0] = 1.0
    overlay[m > 0.5, 1] *= 0.3
    overlay[m > 0.5, 2] *= 0.3

    axes[row, 0].imshow(ch0, cmap="inferno")
    axes[row, 0].set_title(f"#{int(i)} input ch0")
    axes[row, 1].imshow(m, cmap="gray", vmin=0, vmax=1)
    axes[row, 1].set_title("mask")
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title("overlay")
    for ax in axes[row]:
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Все каналы одного сэмпла + маска
i = int(idxs2[0])
img, mask = ds[i]
C = img.shape[0]
fig, axes = plt.subplots(1, C + 1, figsize=(2.2 * (C + 1), 2.4))
for c in range(C):
    axes[c].imshow(img[c].numpy(), cmap="inferno")
    axes[c].set_title(f"ch{c}")
    axes[c].axis("off")
axes[-1].imshow(mask[0].numpy(), cmap="gray", vmin=0, vmax=1)
axes[-1].set_title("mask")
axes[-1].axis("off")
fig.suptitle(f"sample #{i}: all {C} channels", y=1.05)
plt.tight_layout()
plt.show()
